# Limpieza de columnas y preparación para modelado - Accidentes Fatality
Este notebook realiza la limpieza de datos y conserva las columnas más relevantes para un modelo predictivo basado en el dataset de accidentes fatales.

## 1. Importar librerías y configurar el entorno

In [82]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
print("✓ Librerías cargadas correctamente")


✓ Librerías cargadas correctamente


## 2. Cargar el dataset de Accidentes Fatality

In [83]:
data_path = Path('../CSVS/accidentes-fatality-histo.csv')
print('Ruta de datos:', data_path.resolve())
df = pd.read_csv(data_path)
print('Forma original:', df.shape)
df.head(5)


Ruta de datos: C:\Users\Iván\Documents\FP\clase\Proyecto\proyecto-carreteras-sf-fantastic-four\Accidentes_Fatality\CSVS\accidentes-fatality-histo.csv
Forma original: (359, 28)


,unique_id,case_id_fkey,latitude,longitude,collision_year,death_date,death_time,death_datetime,collision_date,collision_time,...,in_coc_2018,publish,on_vz_hin_2022,in_epa_2021,point,analysis_neighborhood,supervisor_district,police_district,data_as_of,data_loaded_at
0,1,140236301,37.710409,-122.404226,2014,2014/03/20,11:21:00,2014/03/20 11:21:00 AM,2014/03/20,NaN,...,False,True,True,False,POINT (-122.404226037 37.710409217),Bayview Hunters Point,10.0,INGLESIDE,2024/09/18 12:00:00 AM,2026/05/01 12:30:54 PM
1,2,140755533,37.725476,-122.394243,2014,2014/09/08,16:38:00,2014/09/08 04:38:00 PM,2014/09/08,05:10:00,...,True,True,True,True,POINT (-122.394243493 37.72547565),Bayview Hunters Point,10.0,BAYVIEW,2024/09/18 12:00:00 AM,2026/05/01 12:30:54 PM
2,4,140365546,37.748255,-122.413669,2014,2014/05/03,17:20:00,2014/05/03 05:20:00 PM,2014/05/03,02:24:00,...,False,True,True,False,POINT (-122.413668844 37.748255329),Mission,9.0,MISSION,2024/09/18 12:00:00 AM,2026/05/01 12:30:54 PM
3,16,150562049,37.777300,-122.419694,2015,2015/06/30,06:00:00,2015/06/30 06:00:00 AM,2015/06/28,03:52:00,...,False,True,True,True,POINT (-122.419693566 37.777299856),Tenderloin,5.0,NORTHERN,2024/09/18 12:00:00 AM,2026/05/01 12:30:54 PM
4,17,140104811,37.778251,-122.419883,2014,2014/02/06,10:20:00,2014/02/06 10:20:00 AM,2014/02/05,02:26:00,...,False,True,True,True,POINT (-122.419883231 37.778251017),Hayes Valley,5.0,NORTHERN,2024/09/18 12:00:00 AM,2026/05/01 12:30:54 PM


## 3. Inspección inicial de columnas y tipos

In [84]:
print('Columnas:')
print(df.columns.tolist())
print('Tipos de datos:')
print(df.dtypes)
print('Valores nulos por columna:')
print(df.isnull().sum().sort_values(ascending=False))


Columnas:
['unique_id', 'case_id_fkey', 'latitude', 'longitude', 'collision_year', 'death_date', 'death_time', 'death_datetime', 'collision_date', 'collision_time', 'collision_datetime', 'location', 'age', 'sex', 'deceased', 'collision_type', 'street_type', 'on_vz_hin_2017', 'in_coc_2018', 'publish', 'on_vz_hin_2022', 'in_epa_2021', 'point', 'analysis_neighborhood', 'supervisor_district', 'police_district', 'data_as_of', 'data_loaded_at']
Tipos de datos:
unique_id                  int64
case_id_fkey                 str
latitude                 float64
longitude                float64
collision_year             int64
death_date                   str
death_time                   str
death_datetime               str
collision_date               str
collision_time               str
collision_datetime           str
location                     str
age                      float64
sex                          str
deceased                     str
collision_type               str
street_type  

## 4. Columnas importantes para un modelo predictivo

Para modelado, se conservan las columnas que aportan información directa sobre la colisión, el actor y la ubicación.
- Fechas y tiempos: `collision_date`, `collision_datetime`, `collision_year`, `collision_month`, `collision_hour`
- Ubicación: `latitude`, `longitude`, `analysis_neighborhood`, `supervisor_district`
- Tipo de colisión y vehículo: `collision_type`, `street_type`
- Información de la persona: `age`, `sex`, `deceased`
Se eliminan del dataset: `police_district`, `on_vz_hin_2017`, `in_coc_2018`, `publish`, `on_vz_hin_2022`, `in_epa_2021`.
El resto de las columnas se conserva en el orden original del archivo lo más posible.

## 5. Conversión de fechas y creación de variables derivadas

In [85]:
df['collision_date'] = pd.to_datetime(df['collision_date'], errors='coerce')
df['collision_datetime'] = pd.to_datetime(df['collision_datetime'], errors='coerce')
df['death_datetime'] = pd.to_datetime(df['death_datetime'], errors='coerce')
df['collision_year'] = pd.to_numeric(df['collision_year'], errors='coerce').astype('Int64')
df['collision_month'] = df['collision_date'].dt.month.astype('Int64')
df['collision_day'] = df['collision_date'].dt.day.astype('Int64')
df['collision_hour'] = pd.to_datetime(df['collision_time'], format='%H:%M:%S', errors='coerce').dt.hour.astype('Int64')
df['death_year'] = df['death_datetime'].dt.year.astype('Int64')
print('Fechas convertidas y variables derivadas creadas')
print(df[['collision_date', 'collision_datetime', 'collision_year', 'collision_month', 'collision_day', 'collision_hour']].head())


Fechas convertidas y variables derivadas creadas
  collision_date  collision_datetime  collision_year  collision_month  \
0     2014-03-20 2014-03-20 00:00:00            2014                3   
1     2014-09-08 2014-09-08 05:10:00            2014                9   
2     2014-05-03 2014-05-03 02:24:00            2014                5   
3     2015-06-28 2015-06-28 03:52:00            2015                6   
4     2014-02-05 2014-02-05 02:26:00            2014                2   

   collision_day  collision_hour  
0             20            <NA>  
1              8               5  
2              3               2  
3             28               3  
4              5               2  


## 6. Normalización de columnas categóricas y booleanas

In [86]:
df['location'] = df['location'].astype(str).str.strip()
df['sex'] = df['sex'].astype(str).str.strip().str.capitalize().replace({'FemalE': 'Female', 'male': 'Male'})
df['deceased'] = df['deceased'].astype(str).str.strip()
df['collision_type'] = df['collision_type'].astype(str).str.strip()
df['street_type'] = df['street_type'].astype(str).str.strip()
print('Normalización completada para columnas textuales seleccionadas')
df[['location', 'sex', 'deceased', 'collision_type', 'street_type']].head()


Normalización completada para columnas textuales seleccionadas


,location,sex,deceased,collision_type,street_type
0,Bayshore Blvd near Visitation Ave,Female,Pedestrian,Pedestrian vs Motor Vehicle,City Street
1,3rd St at Carrol Ave,Male,Pedestrian,Pedestrian vs LRV,City Street
2,Folsom St and Caesar Chavez,Male,Driver,Motor Vehicle Collision,City Street
3,Van Ness Ave and Hayes St,Male,Motorcyclist,Motorcycle vs Motor Vehicle,City Street
4,Van Ness Ave and Grove St,Male,Pedestrian,Pedestrian vs Motor Vehicle,City Street


## 7. Selección de columnas limpias para el archivo final

In [87]:
excluded_columns = {
    'police_district', 'on_vz_hin_2017', 'in_coc_2018',
    'publish', 'on_vz_hin_2022', 'in_epa_2021', 'collision_year', 
    'death_date', 'death_time', 'collision_date', 'collision_time',
    'point', 'data_as_of', 'data_loaded_at', 'collision_month', 'collision_day', 
    'collision_hour', 'death_year'
}
keep_columns = [col for col in df.columns if col not in excluded_columns]
df_clean = df[keep_columns].copy()
print('Columnas conservadas para el dataset limpio:')
print(df_clean.columns.tolist())
print('Forma del dataset limpio:', df_clean.shape)


Columnas conservadas para el dataset limpio:
['unique_id', 'case_id_fkey', 'latitude', 'longitude', 'death_datetime', 'collision_datetime', 'location', 'age', 'sex', 'deceased', 'collision_type', 'street_type', 'analysis_neighborhood', 'supervisor_district']
Forma del dataset limpio: (359, 14)


## 8. Detección de duplicados y manejo de filas repetidas

In [88]:
print('Duplicados únicos por unique_id:', df_clean['unique_id'].duplicated().sum())
df_clean = df_clean.drop_duplicates(subset=['unique_id'])
df_clean = df_clean.sort_values(by='unique_id', key=pd.to_numeric, na_position='last').reset_index(drop=True)
print('Forma después de eliminar duplicados y ordenar por unique_id:', df_clean.shape)


Duplicados únicos por unique_id: 0
Forma después de eliminar duplicados y ordenar por unique_id: (359, 14)


## 9. Revisión de valores nulos en el dataset limpio

In [89]:
null_summary = pd.DataFrame({
    'Columna': df_clean.columns,
    'Valores Nulos': df_clean.isnull().sum().values,
    'Porcentaje (%)': (df_clean.isnull().sum().values / len(df_clean) * 100).round(2)
})
print(null_summary.sort_values('Valores Nulos', ascending=False).to_string(index=False))


              Columna  Valores Nulos  Porcentaje (%)
         case_id_fkey              7            1.95
                  age              1            0.28
  supervisor_district              1            0.28
analysis_neighborhood              1            0.28
             latitude              0            0.00
            unique_id              0            0.00
   collision_datetime              0            0.00
       death_datetime              0            0.00
            longitude              0            0.00
             location              0            0.00
             deceased              0            0.00
                  sex              0            0.00
          street_type              0            0.00
       collision_type              0            0.00


## 10. Guardar el dataset limpio

In [90]:
output_path = Path('../CSVS/accidentes-fatality-histo-cleaned.csv')
df_clean.to_csv(output_path, index=False)
print('Dataset limpio guardado en:', output_path.resolve())


Dataset limpio guardado en: C:\Users\Iván\Documents\FP\clase\Proyecto\proyecto-carreteras-sf-fantastic-four\Accidentes_Fatality\CSVS\accidentes-fatality-histo-cleaned.csv
